# Template Matching: Essential Validation & Enhancement Study

This notebook evaluates the **4 essential configurations** of the Template Matching pipeline on the **139 validation PCB images** (`data/dataset_split.csv`).

### Enhancement Strategy:
1. **Multi-Scale Window Exploration**: $32\times32$ vs $64\times64$ vs $96\times96$.
2. **Adaptive Contour Merging (`cv2.findContours`)**: Fusing overlapping sliding blocks into unified, tightly-bounding defect polygons.
3. **Correlation Threshold Sweep**: Balancing precision and recall ($
ho = 0.55$ vs $0.65$).

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import yaml

PROJECT_ROOT = Path.cwd().parent.parent if Path.cwd().name == 'validation' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from algorithms.common import load_image, preprocess_pair
from algorithms.evaluation import evaluate_boxes, parse_voc_boxes
from algorithms.template_matching import detect_template_matching
from algorithms.preprocessing import build_preprocessing_config

plt.rcParams['figure.dpi'] = 110
print(f'Project Root: {PROJECT_ROOT}')

# Smart auto-run fallback: auto-compute benchmark if outputs/metrics CSV is missing
res_csv = PROJECT_ROOT / 'outputs' / 'metrics' / 'essential_validation_comparison.csv'
if not res_csv.exists():
    print('⚠️ Validation metrics CSV not found. Auto-running validation benchmark on 139 validation images...')
    from scripts.run_essential_validation import main as run_benchmark
    run_benchmark()
    print('✅ Benchmark successfully generated!')


## 1. Essential Validation Results (4 Core Configurations)

In [ ]:
df = pd.read_csv(res_csv)
tm_df = df[df['algorithm'] == 'Template Matching'][['combination_id', 'description', 'precision', 'recall', 'f1_score', 'mean_runtime_ms']]
display(tm_df)


## 2. Visual Comparison of Template Matching Configurations

In [ ]:
plt.figure(figsize=(9, 4))
plt.barh(tm_df['combination_id'], tm_df['f1_score'], color='#3498db')
plt.xlabel('Validation F1-Score (IoU >= 0.50)')
plt.title('Template Matching: F1-Score across 4 Essential Configurations')
plt.xlim(0, 0.60)
for i, v in enumerate(tm_df['f1_score']):
    plt.text(v + 0.01, i, f'{v:.4f}', va='center', fontweight='bold')
plt.tight_layout()
plt.show()


## 3. Visual Detection Overlay Across Defect Classes

In [ ]:
with open(PROJECT_ROOT / 'configs' / 'frozen_parameters.yaml') as f:
    cfg = yaml.safe_load(f)
prep_cfg = build_preprocessing_config(cfg.get('preprocessing'))

manifest = pd.read_csv(PROJECT_ROOT / 'data' / 'dataset_split.csv')
val_subset = manifest[manifest['split'] == 'validation']

sample_classes = ['missing_hole', 'mouse_bite', 'open_circuit', 'short_circuit']
fig, axes = plt.subplots(len(sample_classes), 3, figsize=(15, 4 * len(sample_classes)))

for i, cls in enumerate(sample_classes):
    matching_rows = val_subset[val_subset['defect_class'] == cls]
    if matching_rows.empty:
        continue
    row = matching_rows.iloc[0]
    ref = load_image(PROJECT_ROOT / row['reference_path'])
    def_img = load_image(PROJECT_ROOT / row['image_path'])
    gt_boxes = parse_voc_boxes(PROJECT_ROOT / row['annotation_path'])
    
    det = detect_template_matching(ref, def_img, block_size=(32, 32), step_size=16, corr_threshold=0.65, preprocessing_config=prep_cfg)
    
    overlay = cv2.cvtColor(def_img.copy(), cv2.COLOR_BGR2RGB)
    for b in gt_boxes:
        cv2.rectangle(overlay, (int(b['xmin']), int(b['ymin'])), (int(b['xmax']), int(b['ymax'])), (0, 255, 0), 4)
    for b in det.boxes:
        cv2.rectangle(overlay, (int(b['xmin']), int(b['ymin'])), (int(b['xmax']), int(b['ymax'])), (255, 0, 0), 3)
    
    axes[i, 0].imshow(cv2.cvtColor(ref, cv2.COLOR_BGR2RGB))
    axes[i, 0].set_title(f'{cls} - Reference', fontsize=11)
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(cv2.cvtColor(def_img, cv2.COLOR_BGR2RGB))
    axes[i, 1].set_title(f"{row['image_id']} - Defective", fontsize=11)
    axes[i, 1].axis('off')
    
    axes[i, 2].imshow(overlay)
    axes[i, 2].set_title(f'Detections (Red: {len(det.boxes)}) vs GT (Green: {len(gt_boxes)})', fontsize=11)
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()
